# Notebook 05 – Model Tuning and Final Model

## Part A – Shared Tuning Setup and Random Forest

This notebook performs hyperparameter tuning for the selected candidate models using the approved preprocessing pipeline and a fixed train/validation/test split.

### Part A Objectives
- Recreate the shared dataset and split from Notebook 04.
- Load and verify the Notebook 04 handoff reports.
- Define the shared 5-fold stratified cross-validation setup.
- Tune the Random Forest classifier using RandomizedSearchCV.
- Evaluate all sampled Random Forest configurations.
- Analyze cross-validation performance and overfitting.
- Evaluate the tuned Random Forest on the validation set.
- Export the Random Forest tuning results for the team handoff.

The reserved test set is not used during this tuning stage.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

# Find the project root and add it to Python's import path
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import PREDICTIVE_FEATURES, build_preprocessor

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

print("Project root:", ROOT)
print("src import: OK")

Project root: d:\Obesity-Risk-Intelligence-System
src import: OK


In [3]:
DATA = ROOT / "data"
REPORTS = ROOT / "reports" / "generated"

SEED = 42
TARGET = "NObeyesdad"

CLASSES = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

print("ROOT:", ROOT)
print("DATA:", DATA)
print("REPORTS:", REPORTS)
print("SEED:", SEED)
print("TARGET:", TARGET)

ROOT: d:\Obesity-Risk-Intelligence-System
DATA: d:\Obesity-Risk-Intelligence-System\data
REPORTS: d:\Obesity-Risk-Intelligence-System\reports\generated
SEED: 42
TARGET: NObeyesdad


## 1. Load and verify the dataset

The approved dataset contains 20,758 labelled observations and 18 columns:

- `id`
- 16 approved predictive features
- `NObeyesdad` as the seven-class target

The `id` column is excluded from modelling, and BMI is not included in the approved predictor set.

In [4]:
frame = pd.read_csv(DATA / "raw" / "obesity.csv")

print("Shape:", frame.shape)
print("Columns:")
print(frame.columns.tolist())

Shape: (20758, 18)
Columns:
['id', 'Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']


In [5]:
X = frame[list(PREDICTIVE_FEATURES)].copy()
y = frame[TARGET].copy()

assert X.shape == (20758, 16)
assert y.nunique() == 7

assert "id" not in X.columns
assert "BMI" not in X.columns

assert list(y.unique())  # confirms target exists

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of predictors:", X.shape[1])
print("Number of classes:", y.nunique())
print("\nClasses:")
print(sorted(y.unique()))

X shape: (20758, 16)
y shape: (20758,)
Number of predictors: 16
Number of classes: 7

Classes:
['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']


## 2. Recreate the fixed train/validation/test split

The same two-stage stratified split from Notebook 04 is reproduced using random seed 42.

The resulting partitions are:

- Training: 14,530 observations
- Validation: 3,114 observations
- Reserved test: 3,114 observations

Stratification is used to preserve the seven-class distribution across the partitions.

The test set remains untouched during hyperparameter tuning.

In [6]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    stratify=y_tmp,
    random_state=SEED,
)

assert (len(X_train), len(X_val), len(X_test)) == (
    14530,
    3114,
    3114,
)

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("No index overlap: PASS")

Train: (14530, 16)
Validation: (3114, 16)
Test: (3114, 16)
No index overlap: PASS


## 3. Shared evaluation metrics

Four metrics are used throughout the model comparison:

- Accuracy
- Balanced Accuracy
- Macro F1
- Weighted F1

Macro F1 is used as the primary model-selection metric during hyperparameter tuning because it gives equal importance to all seven classes.

In [7]:
def scores(name, actual, predicted):
    return {
        "Model": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Balanced Accuracy": balanced_accuracy_score(actual, predicted),
        "Macro F1": f1_score(
            actual,
            predicted,
            average="macro",
            zero_division=0,
        ),
        "Weighted F1": f1_score(
            actual,
            predicted,
            average="weighted",
            zero_division=0,
        ),
    }

## 4. Verify the Notebook 04 handoff

Notebook 04 produced two reports:

1. `baseline_model_comparison.csv`
2. `tuning_candidates.csv`

The first contains the five baseline models, while the second identifies the two models selected for hyperparameter tuning.

In [8]:
baseline_report = pd.read_csv(
    REPORTS / "baseline_model_comparison.csv"
)

candidate_report = pd.read_csv(
    REPORTS / "tuning_candidates.csv"
)

assert len(baseline_report) == 5
assert len(candidate_report) == 2

assert set(candidate_report["Model"]) == {
    "Random Forest",
    "Gradient Boosting",
}

print("Baseline report:")
display(baseline_report)

print("\nTuning candidates:")
display(candidate_report)

Baseline report:


,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1,Macro F1 Rank,Balanced Accuracy Rank,Accuracy Rank
0,Gradient Boosting,0.903340,0.893142,0.893031,0.902952,1,1,1
1,Random Forest,0.895633,0.884543,0.884858,0.895254,2,2,2
2,Logistic Regression,0.858060,0.843242,0.841809,0.856599,3,3,3
3,Decision Tree,0.845215,0.831168,0.830801,0.845645,4,4,4
4,Dummy Classifier,0.194926,0.142857,0.046608,0.063596,5,5,5



Tuning candidates:


,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1,Training Macro F1,Macro F1 Gap
0,Gradient Boosting,0.903340,0.893142,0.893031,0.902952,0.912625,0.019594
1,Random Forest,0.895633,0.884543,0.884858,0.895254,1.000000,0.115142


## 5. Shared five-fold cross-validation setup

A five-fold StratifiedKFold strategy is used for hyperparameter tuning.

The same configuration must be reused for both Random Forest and Gradient Boosting so that the two tuning processes remain directly comparable.

For each model, 20 hyperparameter configurations are sampled and evaluated across five folds, resulting in 100 cross-validation fits per model.

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
}

SEARCH_CONFIG = dict(
    n_iter=20,
    scoring=SCORING,
    refit="macro_f1",
    cv=cv,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

print("CV:", cv)
print("Search configuration:")
print(SEARCH_CONFIG)

CV: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Search configuration:
{'n_iter': 20, 'scoring': {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'macro_f1': 'f1_macro', 'weighted_f1': 'f1_weighted'}, 'refit': 'macro_f1', 'cv': StratifiedKFold(n_splits=5, random_state=42, shuffle=True), 'random_state': 42, 'n_jobs': -1, 'verbose': 1, 'return_train_score': True}


## 6. Random Forest hyperparameter search space

The Random Forest search explores six hyperparameters:

- `n_estimators`: number of trees
- `max_depth`: maximum tree depth
- `min_samples_split`: minimum samples required to split an internal node
- `min_samples_leaf`: minimum samples required in a leaf
- `max_features`: number of features considered when finding a split
- `class_weight`: optional class balancing strategy

RandomizedSearchCV samples 20 combinations from this search space rather than exhaustively evaluating every possible combination.

In [10]:
rf_params = {
    "classifier__n_estimators": [150, 200, 300, 400],
    "classifier__max_depth": [None, 10, 15, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", 0.5],
    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
    ],
}

print("Random Forest hyperparameters:")

for parameter, values in rf_params.items():
    print(f"{parameter}: {values}")

Random Forest hyperparameters:
classifier__n_estimators: [150, 200, 300, 400]
classifier__max_depth: [None, 10, 15, 20, 30]
classifier__min_samples_split: [2, 5, 10]
classifier__min_samples_leaf: [1, 2, 4]
classifier__max_features: ['sqrt', 'log2', 0.5]
classifier__class_weight: [None, 'balanced', 'balanced_subsample']


## 7. Build the Random Forest pipeline

The approved preprocessing pipeline is placed directly inside the model pipeline.

This ensures that preprocessing is learned only from the training folds during cross-validation, preventing preprocessing leakage.

The Random Forest uses `n_jobs=1` because the outer RandomizedSearchCV already uses parallel processing with `n_jobs=-1`.

In [11]:
rf_pipe = Pipeline([
    (
        "preprocessor",
        build_preprocessor(),
    ),
    (
        "classifier",
        RandomForestClassifier(
            random_state=SEED,
            n_jobs=1,
        ),
    ),
])

print(rf_pipe)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Height', 'Weight',
                                                   'FCVC', 'NCP', 'CH2O', 'FAF',
                                                   'TUE']),
                                                 ('ordinal',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
              

## 8. Random Forest hyperparameter tuning

RandomizedSearchCV is fitted using only the training partition.

The validation and test partitions are not used during the search.

The model is selected using cross-validated Macro F1, while Accuracy, Balanced Accuracy, and Weighted F1 are also recorded.

In [12]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_params,
    **SEARCH_CONFIG,
)

rf_search.fit(X_train, y_train)

print("\nBest CV Macro F1:", rf_search.best_score_)

print("\nBest parameters:")
for parameter, value in rf_search.best_params_.items():
    print(f"{parameter}: {value}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best CV Macro F1: 0.887985191412794

Best parameters:
classifier__n_estimators: 200
classifier__min_samples_split: 5
classifier__min_samples_leaf: 1
classifier__max_features: 0.5
classifier__max_depth: 30
classifier__class_weight: balanced_subsample


In [13]:
print("Search completed.")
print("Number of candidates evaluated:", len(rf_search.cv_results_))
print("Best CV Macro F1:", rf_search.best_score_)
print(
    "Best estimator:",
    type(rf_search.best_estimator_).__name__,
)

Search completed.
Number of candidates evaluated: 71
Best CV Macro F1: 0.887985191412794
Best estimator: Pipeline


## 9. Inspect Random Forest cross-validation results

All 20 sampled configurations are retained for analysis.

The cross-validation Macro F1 gap is calculated as:

Training Macro F1 − Validation Fold Macro F1

A larger positive gap can indicate that a configuration fits the training folds substantially better than unseen validation folds, which may indicate overfitting.

In [14]:
rf_full = pd.DataFrame(rf_search.cv_results_)

rf_full["CV Macro F1 Gap"] = (
    rf_full["mean_train_macro_f1"]
    - rf_full["mean_test_macro_f1"]
)

display(
    rf_full.sort_values("rank_test_macro_f1")[
        [
            "rank_test_macro_f1",
            "mean_test_macro_f1",
            "std_test_macro_f1",
            "mean_train_macro_f1",
            "CV Macro F1 Gap",
        ]
    ].head(10)
)

,rank_test_macro_f1,mean_test_macro_f1,std_test_macro_f1,mean_train_macro_f1,CV Macro F1 Gap
6,1,0.887985,0.006205,0.984675,0.096690
4,2,0.887369,0.006628,0.984512,0.097143
16,3,0.887195,0.006269,0.945690,0.058495
19,4,0.886837,0.005373,0.990737,0.103900
11,5,0.886801,0.006896,0.964226,0.077425
5,6,0.885713,0.006868,0.973375,0.087663
10,7,0.884659,0.006119,0.959940,0.075281
15,8,0.883115,0.007370,0.988821,0.105706
14,9,0.881501,0.005342,0.974817,0.093317
3,10,0.880420,0.006024,0.940148,0.059728


## 10. Validation evaluation of the tuned Random Forest

The best Random Forest found through cross-validation is now evaluated on the separate validation partition.

This validation result is compared with the original Notebook 04 Random Forest baseline.

The reserved test set is still not evaluated in Part A.

In [15]:
rf_val = scores(
    "Random Forest",
    y_val,
    rf_search.best_estimator_.predict(X_val),
)

base_rf = baseline_report.set_index("Model").loc[
    "Random Forest",
    "Macro F1",
]

print("Baseline RF Macro F1:", base_rf)
print("Tuned RF Macro F1:", rf_val["Macro F1"])
print("Change:", rf_val["Macro F1"] - base_rf)

print("\nTuned Random Forest validation metrics:")
display(pd.DataFrame([rf_val]))

Baseline RF Macro F1: 0.8848580437360324
Tuned RF Macro F1: 0.8982369286279751
Change: 0.013378884891942722

Tuned Random Forest validation metrics:


,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
0,Random Forest,0.907193,0.89844,0.898237,0.907053


## 11. Export Random Forest tuning results

The complete results from all 20 sampled Random Forest configurations are exported to:

`reports/generated/random_forest_tuning_results.csv`

The exported report contains the ranking, cross-validation metrics, training Macro F1, fit time, and all six Random Forest hyperparameters.

In [16]:
cv_cols = [
    "rank_test_macro_f1",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_train_macro_f1",
    "mean_test_accuracy",
    "mean_test_balanced_accuracy",
    "mean_test_weighted_f1",
    "mean_fit_time",
]

rf_cols = cv_cols + [
    "param_" + parameter
    for parameter in rf_params
]

rf_report = rf_full.sort_values(
    "rank_test_macro_f1"
)[rf_cols]

rf_report.to_csv(
    REPORTS / "random_forest_tuning_results.csv",
    index=False,
)

assert len(rf_report) == 20

print(
    "Saved:",
    REPORTS / "random_forest_tuning_results.csv",
)

print("Rows exported:", len(rf_report))
print("Columns exported:", len(rf_report.columns))

display(rf_report.head())

Saved: d:\Obesity-Risk-Intelligence-System\reports\generated\random_forest_tuning_results.csv
Rows exported: 20
Columns exported: 14


,rank_test_macro_f1,mean_test_macro_f1,std_test_macro_f1,mean_train_macro_f1,mean_test_accuracy,mean_test_balanced_accuracy,mean_test_weighted_f1,mean_fit_time,param_classifier__n_estimators,param_classifier__max_depth,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_features,param_classifier__class_weight
6,1,0.887985,0.006205,0.984675,0.898899,0.887953,0.898777,11.124273,200,30,5,1,0.5,balanced_subsample
4,2,0.887369,0.006628,0.984512,0.898417,0.887320,0.898240,16.605994,300,30,5,1,0.5,balanced_subsample
16,3,0.887195,0.006269,0.945690,0.898348,0.887002,0.898087,7.865835,150,20,10,2,0.5,NaN
19,4,0.886837,0.005373,0.990737,0.898004,0.886655,0.897763,10.241822,200,15,2,1,0.5,NaN
11,5,0.886801,0.006896,0.964226,0.897729,0.887050,0.897694,8.235486,150,15,5,2,0.5,balanced


## 12. Part A completion checks

The following checks confirm that the Random Forest tuning workflow completed successfully.

- The shared split remains unchanged.
- Exactly 20 Random Forest configurations were evaluated.
- Macro F1 was used for model selection.
- The validation set was used only after tuning.
- The test set was not used for tuning or validation comparison.
- The complete Random Forest tuning report was exported.

In [17]:
assert len(rf_search.cv_results_["params"]) == 20
assert rf_search.refit == "macro_f1"
assert len(rf_report) == 20
assert set(rf_report.columns) == set(rf_cols)

assert len(X_train) == 14530
assert len(X_val) == 3114
assert len(X_test) == 3114

print("20 RF configurations: PASS")
print("Macro F1 refit: PASS")
print("RF tuning report: PASS")
print("Train/validation/test split: PASS")
print("Part A Random Forest tuning workflow: COMPLETE")

20 RF configurations: PASS
Macro F1 refit: PASS
RF tuning report: PASS
Train/validation/test split: PASS
Part A Random Forest tuning workflow: COMPLETE


## 13. Part A handoff to Part B

### Random Forest objects retained in the notebook

The following objects contain the Random Forest tuning results:

- `rf_search` – fitted RandomizedSearchCV object
- `rf_full` – complete 20-configuration CV results
- `rf_val` – tuned Random Forest validation metrics
- `rf_report` – exported Random Forest tuning report
- `SEARCH_CONFIG` – shared search configuration
- `cv` – shared StratifiedKFold configuration

### Exported report

`reports/generated/random_forest_tuning_results.csv`

Part B can use the same shared data split, scoring configuration, and search configuration for Gradient Boosting.

The reserved test set remains untouched until the joint final-model review.

## 14. Final Random Forest tuning summary

The Random Forest hyperparameter search evaluated 20 sampled configurations using five-fold stratified cross-validation.

The best configuration was selected using cross-validated Macro F1. The selected model was then evaluated on the separate validation partition and compared against the Notebook 04 Random Forest baseline.

The reserved test partition was not used during this tuning stage.

In [19]:
print("===== RANDOM FOREST TUNING SUMMARY =====")

print(f"Best CV Macro F1: {rf_search.best_score_:.6f}")

print("\nBest parameters:")
for parameter, value in rf_search.best_params_.items():
    print(f"  {parameter}: {value}")

print("\nValidation metrics:")
for metric, value in rf_val.items():
    if metric != "Model":
        print(f"  {metric}: {value:.6f}")

print("\nBaseline RF Macro F1:", f"{base_rf:.6f}")
print("Tuned RF Macro F1:", f"{rf_val['Macro F1']:.6f}")
print(
    "Validation Macro F1 change:",
    f"{rf_val['Macro F1'] - base_rf:+.6f}"
)

===== RANDOM FOREST TUNING SUMMARY =====
Best CV Macro F1: 0.887985

Best parameters:
  classifier__n_estimators: 200
  classifier__min_samples_split: 5
  classifier__min_samples_leaf: 1
  classifier__max_features: 0.5
  classifier__max_depth: 30
  classifier__class_weight: balanced_subsample

Validation metrics:
  Accuracy: 0.907193
  Balanced Accuracy: 0.898440
  Macro F1: 0.898237
  Weighted F1: 0.907053

Baseline RF Macro F1: 0.884858
Tuned RF Macro F1: 0.898237
Validation Macro F1 change: +0.013379


## 15. Random Forest cross-validation performance

The table below shows the strongest Random Forest configurations according to cross-validated Macro F1.

The training-validation Macro F1 gap is retained to help identify configurations that may show signs of overfitting.

In [20]:
top_rf = rf_full.sort_values(
    "rank_test_macro_f1"
).copy()

top_rf[
    [
        "rank_test_macro_f1",
        "mean_test_macro_f1",
        "std_test_macro_f1",
        "mean_train_macro_f1",
        "CV Macro F1 Gap",
    ]
].head(10)

,rank_test_macro_f1,mean_test_macro_f1,std_test_macro_f1,mean_train_macro_f1,CV Macro F1 Gap
6,1,0.887985,0.006205,0.984675,0.096690
4,2,0.887369,0.006628,0.984512,0.097143
16,3,0.887195,0.006269,0.945690,0.058495
19,4,0.886837,0.005373,0.990737,0.103900
11,5,0.886801,0.006896,0.964226,0.077425
5,6,0.885713,0.006868,0.973375,0.087663
10,7,0.884659,0.006119,0.959940,0.075281
15,8,0.883115,0.007370,0.988821,0.105706
14,9,0.881501,0.005342,0.974817,0.093317
3,10,0.880420,0.006024,0.940148,0.059728


## 16. Export verification

The Random Forest tuning report contains the 20 sampled configurations and the required cross-validation results and hyperparameters.

In [22]:
rf_output = REPORTS / "random_forest_tuning_results.csv"

assert rf_output.exists()

saved_rf_report = pd.read_csv(rf_output)

assert len(saved_rf_report) == 20
assert len(saved_rf_report.columns) == 14

print("File exists:", rf_output.exists())
print("Rows:", len(saved_rf_report))
print("Columns:", len(saved_rf_report.columns))

display(saved_rf_report.head())

File exists: True
Rows: 20
Columns: 14


,rank_test_macro_f1,mean_test_macro_f1,std_test_macro_f1,mean_train_macro_f1,mean_test_accuracy,mean_test_balanced_accuracy,mean_test_weighted_f1,mean_fit_time,param_classifier__n_estimators,param_classifier__max_depth,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_features,param_classifier__class_weight
0,1,0.887985,0.006205,0.984675,0.898899,0.887953,0.898777,11.124273,200,30.0,5,1,0.5,balanced_subsample
1,2,0.887369,0.006628,0.984512,0.898417,0.887320,0.898240,16.605994,300,30.0,5,1,0.5,balanced_subsample
2,3,0.887195,0.006269,0.945690,0.898348,0.887002,0.898087,7.865835,150,20.0,10,2,0.5,NaN
3,4,0.886837,0.005373,0.990737,0.898004,0.886655,0.897763,10.241822,200,15.0,2,1,0.5,NaN
4,5,0.886801,0.006896,0.964226,0.897729,0.887050,0.897694,8.235486,150,15.0,5,2,0.5,balanced
